In [1]:
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import gradio as gr
from ultralytics import YOLO
import time

Creating new Ultralytics Settings v0.0.8 file  
View Ultralytics Settings with 'yolo settings' or at 'C:\Users\Victus\AppData\Roaming\Ultralytics\settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.


In [4]:
#####################################
# LOAD IMAGES
#####################################

image_1_path = "images/children_1.jpg"
image_2_path = "images/children_2.jpg"
cropped_image_path = "images/chidren_1_cropped.jpg"

In [ ]:
def load_images(scene1_path, scene2_path, template_path):
    scene1 = cv2.imread(scene1_path)
    scene2 = cv2.imread(scene2_path)
    template = cv2.imread(template_path)

    if scene1 is None:
        raise FileNotFoundError(f"Could not load: {scene1_path}")

    if scene2 is None:
        raise FileNotFoundError(f"Could not load: {scene2_path}")

    if template is None:
        raise FileNotFoundError(f"Could not load: {template_path}")

    return scene1, scene2, template

In [ ]:
image_1 = load_image(image_1_path, grayscale=True)
image_2 = load_image(image_2_path, grayscal=True)
cropped_image = load_image(cropped_image_path, grayscale=True)

## Task 1. Template matching and why it is not enough

In [ ]:
#####################################
# RUN TEMPLATE MATCHING
#####################################
def match_template(scene, template):
    score_map = cv2.matchTemplate(
        scene,
        template,
        cv2.TM_CCOEFF_NORMED
    )

    _, max_score, _, max_location = cv2.minMaxLoc(
        score_map
    )

    return score_map, max_score, max_location

In [ ]:
#####################################
# DRAW THE DETECTED TEMPLATE LOCATION
#####################################
def draw_match(scene, template, location):
    result = scene.copy()

    template_height, template_width = template.shape

    top_left = location

    bottom_right = (
        top_left[0] + template_width,
        top_left[1] + template_height
    )

    cv2.rectangle(
        result,
        top_left,
        bottom_right,
        (0, 255, 0),
        2
    )

    return result

In [ ]:
#####################################
# VISUALISE SCORE MAPS
####################################
def plot_score_maps(
    template,
    score_map1,
    score_map2,
    score1,
    score2
):
    plt.figure(figsize=(15, 5))

    # Template
    plt.subplot(1, 3, 1)
    plt.imshow(template, cmap="gray")
    plt.title("Cropped Template")
    plt.axis("off")

    # Original scene score map
    plt.subplot(1, 3, 2)
    plt.imshow(score_map1, cmap="gray")
    plt.title(f"Original Scene\nMax = {score1:.3f}")
    plt.colorbar()
    plt.axis("off")

    # Changed scene score map
    plt.subplot(1, 3, 3)
    plt.imshow(score_map2, cmap="gray")
    plt.title(f"Different Distance/Angle\nMax = {score2:.3f}")
    plt.colorbar()
    plt.axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
#####################################
# VISUALISE DETECTED LOCATIONS
#####################################
def plot_detected_matches(
    scene1,
    scene2,
    match1,
    match2,
    score1,
    score2
):
    plt.figure(figsize=(12, 5))

    plt.subplot(1, 2, 1)
    plt.imshow(
        cv2.cvtColor(match1, cv2.COLOR_BGR2RGB)
    )
    plt.title(f"Original Scene\nScore = {score1:.3f}")
    plt.axis("off")

    plt.subplot(1, 2, 2)
    plt.imshow(
        cv2.cvtColor(match2, cv2.COLOR_BGR2RGB)
    )
    plt.title(
        f"Different Distance/Angle\nScore = {score2:.3f}"
    )
    plt.axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
#####################################
# TEMPLATE MATCHING PIPELINE
#####################################
def template_matching_pipeline(
    scene1_path,
    scene2_path,
    template_path
):

    # Step 1: Load
    scene1, scene2, template = load_images(
        scene1_path,
        scene2_path,
        template_path
    )

    # Step 2: Preprocess
    gray_scene1, gray_scene2, gray_template = (
        preprocess_images(
            scene1,
            scene2,
            template
        )
    )

    # Step 3: Match template in original scene
    score_map1, score1, location1 = match_template(
        gray_scene1,
        gray_template
    )

    # Step 4: Match template in changed scene
    score_map2, score2, location2 = match_template(
        gray_scene2,
        gray_template
    )

    # Step 5: Draw detections
    detected_scene1 = draw_match(
        scene1,
        gray_template,
        location1
    )

    detected_scene2 = draw_match(
        scene2,
        gray_template,
        location2
    )

    # Step 6: Display score maps
    plot_score_maps(
        gray_template,
        score_map1,
        score_map2,
        score1,
        score2
    )

    # Step 7: Display detected locations
    plot_detected_matches(
        scene1,
        scene2,
        detected_scene1,
        detected_scene2,
        score1,
        score2
    )

    # Step 8: Return results for further analysis
    return {
        "original_score": score1,
        "changed_score": score2,
        "original_location": location1,
        "changed_location": location2,
        "original_score_map": score_map1,
        "changed_score_map": score_map2
    }


In [ ]:
#####################################
# RUN THE PIPELINE
#####################################
results = template_matching_pipeline(
    scene1_path="image1.jpg",
    scene2_path="image2.jpg",
    template_path="cropped_object.jpg"
)

print("\nTemplate Matching Results")
print("-" * 30)
print(
    f"Original scene score: "
    f"{results['original_score']:.4f}"
)
print(
    f"Changed scene score: "
    f"{results['changed_score']:.4f}"
)
print(
    f"Original match location: "
    f"{results['original_location']}"
)
print(
    f"Changed match location: "
    f"{results['changed_location']}"
)

## Task 1. Template matching and why it is not enough

## Task 2. Viola-Jones face detection

## Task 3. Modern one-shot detection with YOLO

## Task 4. Your first vision app

## Task 5. Ethics reflection